# Steps 2 & 3: Fetch DocVQA to Colab and Upload to Hugging Face Hub

This notebook executes the ingestion and upload pipeline for **DocVQA Task 1 (Single Page Document VQA)**:
1. **High-Speed Download:** Fetches document images (`spdocvqa_images.tar.gz`), questions & answers (`spdocvqa_qas.zip`), OCR outputs (`spdocvqa_ocr.tar.gz`), and preprocessed IMDBs (`spdocvqa_imdb.zip`) using `aria2c`.
2. **Canonical Structuring:** Organizes assets into `images/`, `annotations/`, `ocr/`, `imdb/`.
3. **Dataset Card Generation:** Automatically generates a comprehensive `README.md` with dataset tags.
4. **Hugging Face Upload:** Uploads via `HfApi.upload_folder(...)`.
5. **Disk Reclaim:** Purges local files to keep Colab VM storage clean.

In [ ]:
# 1. Setup tools and authenticate
!sudo apt-get install -y -qq aria2
!pip install -q --upgrade huggingface_hub

import os
import shutil
from pathlib import Path
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, ModuleNotFoundError):
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found! Please configure HF_TOKEN in Colab Secrets.")

login(token=HF_TOKEN, write_permission=True)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
DOCVQA_REPO = f"{username}/docvqa-task1-spdocvqa"
print(f"✓ Authenticated as: {username}")
print(f"✓ Target Repo: {DOCVQA_REPO}")

In [ ]:
# 2. Check initial disk allocation
!df -h /

In [ ]:
# 3. Download DocVQA Task 1 files via aria2c
DOCVQA_RAW = Path("./docvqa_raw")
DOCVQA_CLEAN = Path("./docvqa_dataset")

shutil.rmtree(DOCVQA_RAW, ignore_errors=True)
shutil.rmtree(DOCVQA_CLEAN, ignore_errors=True)
DOCVQA_RAW.mkdir(parents=True, exist_ok=True)

# Direct backend download URLs from CVC / RRC Challenge 17
docvqa_urls = """
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_qas.zip
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_images.tar.gz
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_ocr.tar.gz
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_imdb.zip
"""

with open("docvqa_urls.txt", "w") as f:
    f.write(docvqa_urls.strip())

print(">>> Starting high-speed parallel download with aria2c...")
!aria2c -j 4 -x 8 -s 8 -k 1M --check-certificate=false -i docvqa_urls.txt --dir={DOCVQA_RAW}

print("Downloaded raw assets:")
!ls -lh {DOCVQA_RAW}

In [ ]:
# 4. Restructure into canonical layout & generate README.md
(DOCVQA_CLEAN / "images").mkdir(parents=True, exist_ok=True)
(DOCVQA_CLEAN / "annotations").mkdir(parents=True, exist_ok=True)
(DOCVQA_CLEAN / "ocr").mkdir(parents=True, exist_ok=True)
(DOCVQA_CLEAN / "imdb").mkdir(parents=True, exist_ok=True)

for file in DOCVQA_RAW.iterdir():
    if "images" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "images" / file.name)
    elif "qas" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "annotations" / file.name)
    elif "ocr" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "ocr" / file.name)
    elif "imdb" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "imdb" / file.name)

# Dataset Card
docvqa_readme = f"""---
license: other
task_categories:
- visual-question-answering
size_categories:
- 10K<n<100K
tags:
- docvqa
- document-ai
- vqa
- multimodal
---

# DocVQA (Task 1: Single Page Document VQA) Archive

- **Source Portal:** [Robust Reading Competition (RRC) Challenge 17](https://rrc.cvc.uab.es/?ch=17&com=downloads)
- **Host:** Computer Vision Center (CVC) / IIIT Hyderabad (UCSF Industry Documents Library)
- **Terms:** Hosted purely for non-commercial research and educational purposes.

## Repository Contents
- `images/`: `spdocvqa_images.tar.gz` (~12,767 document images)
- `annotations/`: `spdocvqa_qas.zip` (~50,000 QA pairs across train, val, and test)
- `ocr/`: `spdocvqa_ocr.tar.gz` (Microsoft OCR token representations)
- `imdb/`: `spdocvqa_imdb.zip` (Preprocessed Image Database files for MP-DocVQA framework)
"""

with open(DOCVQA_CLEAN / "README.md", "w") as f:
    f.write(docvqa_readme.strip())

print("Structured directory:")
!ls -lh {DOCVQA_CLEAN}/*

In [ ]:
# 5. Upload folder to Hugging Face Hub
print(f">>> Uploading DocVQA to https://huggingface.co/datasets/{DOCVQA_REPO}...")
api.upload_folder(
    folder_path=str(DOCVQA_CLEAN),
    repo_id=DOCVQA_REPO,
    repo_type="dataset",
    commit_message="Ingest DocVQA Task 1 SP-DocVQA archives, annotations, OCR, and IMDBs"
)
print("✓ Upload completed successfully!")

In [ ]:
# 6. Verify remote files on Hugging Face Hub
print(f"=== Files in https://huggingface.co/datasets/{DOCVQA_REPO} ===")
remote_files = api.list_repo_files(repo_id=DOCVQA_REPO, repo_type="dataset")
for f in remote_files:
    print(f" - {f}")

In [ ]:
# 7. Purge local disk to reclaim space
print(">>> Purging local files...")
shutil.rmtree(DOCVQA_RAW, ignore_errors=True)
shutil.rmtree(DOCVQA_CLEAN, ignore_errors=True)
if os.path.exists("docvqa_urls.txt"):
    os.remove("docvqa_urls.txt")

print("Remaining disk space after purge:")
!df -h /